# SR-DAS — descriptive statistics of the S&P 500 sample (notebook 6)

**Self-contained** (downloads its own data, contains all code, does not need notebooks 1–5).
It produces the numbers that are still missing in the manuscript:

* the **S&P 500 column of Table 1** (trading days, asset returns and volatilities, market return and volatility, average pairwise correlation);
* the two counts in **Section 4.1**: how many tickers were index members at some time after January 2000, and how many of them have no Yahoo price history.

The data steps are **identical to notebook 2** (same membership file, same Yahoo download, same cleaning), so the numbers match the backtest sample.

**Definitions used for stocks** (stocks enter and leave the index, so the industry definitions cannot be used as they are):
* asset statistics use each stock's returns **only on days when it was an index member**, and only stocks with at least 252 such days;
* the average pairwise correlation is computed **per calendar year** over the stocks that were members with a return on every day of that year, and then averaged over the years.

**Runtime:** 5–15 minutes, almost all of it the Yahoo download. If you kept the notebook-2 price cache on Google Drive, set `SAVE_TO_DRIVE = True` and point `DRIVE_DIR` to the same folder; the download is then skipped.

**At the end:** copy the printed block *"Numbers for the manuscript"* and send it back (or send `outputs_sp500_desc.zip`).

## 0. Setup

In [ ]:
!pip -q install --upgrade yfinance > /dev/null

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)

DATA_START = "2000-01-01"     # same as notebook 2
TEST_START = "2005-01-01"     # first out-of-sample date of the stock backtest
DATA_END = None               # None = up to today
SAVE_TO_DRIVE = False         # True = reuse the notebook-2 price cache on Google Drive
DRIVE_DIR = "/content/drive/MyDrive/srdas/sp500"
DATA_DIR = "data"
OUT_DIR = "outputs_sp500_desc"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = os.path.join(DRIVE_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)
print("Data folder:", DATA_DIR)

## 1. Data (same steps as notebook 2)
### 1.1 Point-in-time S&P 500 membership

In [ ]:
import os
import io
import time
import requests

MEMBERSHIP_URL = ("https://raw.githubusercontent.com/fja05680/sp500/master/"
                  "S%26P%20500%20Historical%20Components%20%26%20Changes%20(Updated).csv")


def load_sp500_membership(cache_dir="data"):
    """Point-in-time S&P 500 membership (fja05680/sp500, based on Clenow's list + updates).

    Each row is a date on which the index changed, with the full member list on that
    date. Membership on any day = the most recent row on or before that day.
    Returns a Series: date -> list of tickers (exchange-style, e.g. 'BRK.B').
    """
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, "sp500_membership.csv")
    if not os.path.exists(path):
        print("Downloading S&P 500 membership history ...")
        r = requests.get(MEMBERSHIP_URL, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"])
    s = df.set_index("date")["tickers"].map(lambda x: sorted(set(t.strip() for t in x.split(",") if t.strip())))
    return s.sort_index()


def yahoo_symbol(t):
    """Exchange ticker -> Yahoo symbol (share classes use '-', e.g. BRK.B -> BRK-B)."""
    return t.replace(".", "-")


def make_universe_fn(membership):
    """Return f(date) -> list of Yahoo symbols that were S&P 500 members on that date."""
    dates = membership.index

    def universe(date):
        i = dates.searchsorted(pd.Timestamp(date), side="right") - 1
        if i < 0:
            return []
        return [yahoo_symbol(t) for t in membership.iloc[i]]
    return universe


def tickers_ever(membership, start, end=None):
    """All tickers that were members at any time in [start, end]."""
    m = membership.loc[:end] if end else membership
    first = m.index.searchsorted(pd.Timestamp(start), side="right") - 1
    rows = m.iloc[max(first, 0):]
    return sorted({yahoo_symbol(t) for lst in rows for t in lst})


In [ ]:
membership = load_sp500_membership(DATA_DIR)
all_tickers = tickers_ever(membership, DATA_START, DATA_END)
print(f"Membership rows: {len(membership)} ({membership.index[0].date()} to {membership.index[-1].date()})")
print(f"Tickers that were index members at some time since {DATA_START}: {len(all_tickers)}")

### 1.2 Prices from Yahoo Finance (cached, resumable)
If a chunk fails because of rate limits, run the cell again: finished chunks are loaded from the cache.

In [ ]:
def _yf_download(tickers, start, end):
    """Thin wrapper around yfinance (kept separate so it can be swapped or mocked)."""
    import yfinance as yf
    df = yf.download(tickers, start=start, end=end, auto_adjust=True, actions=False,
                     progress=False, threads=True, group_by="column")
    if df is None or len(df) == 0:
        return pd.DataFrame()
    if isinstance(df.columns, pd.MultiIndex):
        close = df["Close"]
    else:
        close = df[["Close"]].rename(columns={"Close": tickers[0]})
    if isinstance(close, pd.Series):
        close = close.to_frame(tickers[0])
    return close


def download_prices(tickers, start, end=None, cache_dir="data/prices", chunk=50, pause=1.5, retries=3):
    """Download adjusted close prices in chunks, caching each chunk to disk.

    Re-running skips chunks already on disk, so an interrupted download (rate limit,
    Colab disconnect) resumes where it stopped. Point cache_dir at Google Drive to
    keep the data between Colab sessions.
    """
    os.makedirs(cache_dir, exist_ok=True)
    tickers = sorted(set(tickers))
    frames = []
    chunks = [tickers[i:i + chunk] for i in range(0, len(tickers), chunk)]
    for ci, ch in enumerate(chunks):
        path = os.path.join(cache_dir, f"chunk_{ci:03d}_{len(tickers)}_{start}.pkl")
        if os.path.exists(path):
            frames.append(pd.read_pickle(path))
            continue
        for attempt in range(retries):
            try:
                px = _yf_download(ch, start, end)
                break
            except Exception as e:  # rate limit / network hiccup
                print(f"  chunk {ci}: attempt {attempt + 1} failed ({e}); retrying")
                time.sleep(pause * 5 * (attempt + 1))
                px = pd.DataFrame()
        px = px.reindex(columns=ch) if len(px) else pd.DataFrame(columns=ch)
        px.to_pickle(path)
        frames.append(px)
        print(f"  chunk {ci + 1}/{len(chunks)}: {px.notna().any().sum()}/{len(ch)} tickers with data")
        time.sleep(pause)
    prices = pd.concat(frames, axis=1).sort_index()
    prices.index = pd.to_datetime(prices.index).tz_localize(None)
    return prices.loc[:, ~prices.columns.duplicated()]


In [ ]:
prices = download_prices(all_tickers + ["^GSPC"], DATA_START, DATA_END,
                         cache_dir=os.path.join(DATA_DIR, "prices"))
no_data = [t for t in all_tickers if t not in prices.columns or prices[t].notna().sum() == 0]
print("Price matrix:", prices.shape)
print(f"Tickers with no Yahoo price history: {len(no_data)} of {len(all_tickers)} "
      f"({len(no_data) / len(all_tickers):.1%})")

### 1.3 Cleaning and returns

In [ ]:
def prices_to_returns(prices, min_share=0.5, spike=0.4, verbose=True):
    """Convert prices to clean daily simple returns.

    * returns computed per ticker without forward-filling across gaps;
    * days on which fewer than `min_share` of the tickers that are trading have
      data are dropped (holiday/feed artefacts);
    * one-day spike-and-reversal errors (|r_t| > spike followed by an opposite move
      that undoes it to within 5%) are set to NaN — a common Yahoo data error.
    Returns (returns, report) where report lists the corrections.
    """
    px = prices.sort_index()
    active = px.notna().sum(axis=1)
    keep = active >= min_share * active.rolling(20, min_periods=1).median()
    px = px[keep]
    rets = px.pct_change(fill_method=None)
    nxt = rets.shift(-1)
    spike_mask = (rets.abs() > spike) & (nxt.abs() > spike / 2) & (np.sign(rets) != np.sign(nxt)) & \
                 (((1 + rets) * (1 + nxt) - 1).abs() < 0.05)
    both = spike_mask | spike_mask.shift(1, fill_value=False)
    n_fix = int(both.values.sum())
    rets = rets.mask(both)
    extreme = rets.stack()
    extreme = extreme[extreme.abs() > 0.5]
    report = dict(days_dropped=int((~keep).sum()), spike_points_removed=n_fix,
                  remaining_moves_over_50pct=len(extreme))
    if verbose:
        print("Cleaning report:", report)
    return rets.iloc[1:], report, extreme


def coverage_audit(membership, returns, rebalance_dates, lookback, min_coverage):
    """For each rebalance date: index members, members with any Yahoo data,
    and members eligible for selection (enough history in the look-back)."""
    uni = make_universe_fn(membership)
    rows = []
    idx = returns.index
    for d in rebalance_dates:
        t = idx.searchsorted(d, side="right") - 1
        mem = uni(d)
        have = [m for m in mem if m in returns.columns]
        win = returns.iloc[max(0, t - lookback + 1): t + 1].reindex(columns=have)
        cov = win.notna().mean()
        elig = int(((cov >= min_coverage) & win.iloc[-1].notna()).sum()) if len(have) else 0
        rows.append(dict(date=d, members=len(mem), with_yahoo_data=int((cov > 0).sum()),
                         eligible=elig))
    df = pd.DataFrame(rows).set_index("date")
    df["coverage_%"] = 100 * df["eligible"] / df["members"]
    return df


In [ ]:
rets_all, clean_report, _ = prices_to_returns(prices)
MKT = rets_all.pop("^GSPC")
mask_m = MKT.notna()
rets, MKT = rets_all[mask_m], MKT[mask_m]
print(f"Returns: {rets.shape[1]} tickers, {rets.index[0].date()} to {rets.index[-1].date()}")

## 2. Descriptive statistics of the test period

In [ ]:

TRADING_DAYS = 252


def membership_mask(membership, index, columns):
    """Boolean matrix (days x tickers): True if the ticker was an S&P 500 member on that day."""
    colpos = {c: j for j, c in enumerate(columns)}
    pos = membership.index.searchsorted(index, side="right") - 1
    arr = np.zeros((len(index), len(columns)), dtype=bool)
    for p in np.unique(pos):
        if p < 0:
            continue
        rows = np.where(pos == p)[0]
        cols = [colpos[yahoo_symbol(t)] for t in membership.iloc[p] if yahoo_symbol(t) in colpos]
        if cols:
            arr[np.ix_(rows, cols)] = True
    return pd.DataFrame(arr, index=index, columns=columns)


def avg_offdiag_corr(X):
    """Average pairwise correlation of the columns of a complete T x N array."""
    sd = X.std(axis=0, ddof=1)
    X = X[:, sd > 1e-12]
    if X.shape[1] < 2 or X.shape[0] < 10:
        return np.nan
    Cm = np.corrcoef(X, rowvar=False)
    n = Cm.shape[0]
    return (Cm.sum() - n) / (n * (n - 1))


def yearly_member_corr(rets, mask, days):
    """For each calendar year: stocks that were members with a return on every selected day of
    that year; average pairwise correlation on those days. Returns a Series by year."""
    out = {}
    for y, idx in rets.index.groupby(rets.index.year).items():
        idx = idx[days.reindex(idx).values]
        if len(idx) < 10:
            continue
        R = rets.loc[idx]
        ok = mask.loc[idx].all() & R.notna().all()
        out[y] = avg_offdiag_corr(R.loc[:, ok].values)
    return pd.Series(out)


In [ ]:
R = rets.loc[TEST_START:]
m = MKT.loc[TEST_START:]
mem = membership_mask(membership, R.index, list(R.columns))
Rm = R.where(mem)                                  # returns only while a member

n_obs = Rm.notna().sum()
keep = n_obs[n_obs >= TRADING_DAYS].index
X = Rm[keep]
ann_ret = np.exp(np.log1p(X).sum() * TRADING_DAYS / X.notna().sum()) - 1   # geometric, member days only
ann_vol = X.std() * np.sqrt(TRADING_DAYS)

all_days = pd.Series(True, index=R.index)
stress_days = m <= m.quantile(0.10)               # same stress definition as Table 1 (whole test period)
corr_all = yearly_member_corr(R, mem, all_days)
corr_stress = yearly_member_corr(R, mem, stress_days)

desc = {
    "Start": str(R.index[0].date()),
    "End": str(R.index[-1].date()),
    "Trading days": len(R),
    "Stocks with >= 252 member days": len(keep),
    "Asset ann. return, median": ann_ret.median(),
    "Asset ann. vol, median": ann_vol.median(),
    "Asset ann. vol, min": ann_vol.min(),
    "Asset ann. vol, 5th pct": ann_vol.quantile(0.05),
    "Asset ann. vol, 95th pct": ann_vol.quantile(0.95),
    "Asset ann. vol, max": ann_vol.max(),
    "Market ann. return (S&P 500 price index)": (1 + m).prod() ** (TRADING_DAYS / len(m)) - 1,
    "Market ann. vol": m.std() * np.sqrt(TRADING_DAYS),
    "Avg pairwise corr, all days (mean of yearly values)": corr_all.mean(),
    "Avg pairwise corr, stress days (mean of yearly values)": corr_stress.mean(),
    "Tickers ever members since 2000": len(all_tickers),
    "Tickers with no Yahoo price history": len(no_data),
    "Share with no Yahoo price history": len(no_data) / len(all_tickers),
}
desc = pd.Series(desc, name="S&P 500 stocks")
display(desc.to_frame())
pd.DataFrame({"all days": corr_all, "stress days": corr_stress}).to_csv(os.path.join(OUT_DIR, "yearly_corr.csv"))
desc.to_csv(os.path.join(OUT_DIR, "T1_descriptive_sp500.csv"))
print("Cleaning report:", clean_report)

## 3. Numbers for the manuscript
Copy everything printed by this cell and send it back.

In [ ]:
p = lambda x: f"{100 * x:.1f}"
print("=== Numbers for the manuscript ===")
print(f"Test period: {desc['Start']} to {desc['End']}")
print(f"Trading days & {desc['Trading days']:,}")
print(f"Median asset return (% p.a.) & {p(desc['Asset ann. return, median'])}")
print(f"Median asset volatility (% p.a.) & {p(desc['Asset ann. vol, median'])}")
print(f"Range of asset volatility, min-max (% p.a.) & {p(desc['Asset ann. vol, min'])}--{p(desc['Asset ann. vol, max'])}")
print(f"Range of asset volatility, 5th-95th pct (% p.a.) & {p(desc['Asset ann. vol, 5th pct'])}--{p(desc['Asset ann. vol, 95th pct'])}")
print(f"Market return (% p.a.) & {p(desc['Market ann. return (S&P 500 price index)'])}")
print(f"Market volatility (% p.a.) & {p(desc['Market ann. vol'])}")
print(f"Average pairwise correlation (all days) & {desc['Avg pairwise corr, all days (mean of yearly values)']:.2f}")
print(f"Average pairwise correlation (stress days) & {desc['Avg pairwise corr, stress days (mean of yearly values)']:.2f}")
print(f"Stocks with >= 252 member days: {desc['Stocks with >= 252 member days']}")
print(f"Tickers ever members since 2000: {desc['Tickers ever members since 2000']}")
print(f"No Yahoo history: {desc['Tickers with no Yahoo price history']} ({100 * desc['Share with no Yahoo price history']:.0f}%)")
print(f"Spike points removed in cleaning: {clean_report['spike_points_removed']}; days dropped: {clean_report['days_dropped']}")

In [ ]:
import shutil
shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
try:
    from google.colab import files
    files.download(OUT_DIR + ".zip")
except Exception:
    print("Saved", OUT_DIR + ".zip")